In [2]:
# Importamos las librerías que necesitamos

# Librerías de extracción de datos
# -----------------------------------------------------------------------
from bs4 import BeautifulSoup
import requests

from selenium import webdriver  # Selenium es una herramienta para automatizar la interacción con navegadores web.
from webdriver_manager.chrome import ChromeDriverManager  # ChromeDriverManager gestiona la instalación del controlador de Chrome.
from selenium.webdriver.common.keys import Keys  # Keys es útil para simular eventos de teclado en Selenium.
from selenium.webdriver.support.ui import Select  # Select se utiliza para interactuar con elementos <select> en páginas web.
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException # Excepciones comunes de selenium que nos podemos encontrar ç
from selenium.webdriver.common.by import By

from time import sleep  # Sleep se utiliza para pausar la ejecución del programa por un número de segundos.

# Librerías para tratamiento de datos
# -----------------------------------------------------------------------
import pandas as pd
import numpy as np
import re

#Librerías para hacer la ELT
import sys
sys.path.append("../../")
from src import soporte_sql as s_sql
from src import metodos_limpieza_df as s_limpieza

#opciones vis

pd.set_option("display.max_columns",None)

%load_ext autoreload
%autoreload 2

Vamos a crear las tablas de la bdd

In [ ]:
conn = s_sql.crear_conexion()
cur = conn.cursor()

# Asegurar que no haya transacciones bloqueadas
conn.rollback()  

crear_tablas = """
    
    DROP TABLE IF EXISTS modelos CASCADE;
    CREATE TABLE modelos (
        marca TEXT,
        modelo TEXT,
        anio INTEGER,
        primary key (marca, modelo, anio),
        tipo_carroceria TEXT,
        link TEXT
    );


    DROP TABLE IF EXISTS caracteristicas;
    CREATE TABLE caracteristicas (
        marca TEXT,
        modelo TEXT,
        anio INTEGER,
        primary key (marca, modelo, anio),
        precio_base NUMERIC,
        precio_max NUMERIC,
        fecha_inicio DATE,
        fecha_fin DATE,
        potencia_minima NUMERIC,
        potencia_maxima NUMERIC,
        lon_0 NUMERIC,
        lon_1 NUMERIC,
        maletero_0 NUMERIC,
        maletero_1 NUMERIC,
        cambio_man INT,
        cambio_aut INT,
        tracc_del INT,
        tracc_total INT,
        tracc_tras INT,
        diesel INT,
        gasolina INT,
        electrificado INT,
        FOREIGN KEY (marca, modelo, anio) REFERENCES modelos (marca, modelo, anio)
    );

    DROP TABLE IF EXISTS reviews;
    CREATE TABLE reviews (
        marca TEXT,
        modelo TEXT,
        anio INTEGER,
        primary key (marca, modelo, anio),
        reportaje TEXT,
        FOREIGN KEY (marca, modelo, anio) REFERENCES modelos (marca, modelo, anio)
    );
"""


try:
    cur.execute(crear_tablas)
    conn.commit()
    print("Tablas creadas correctamente ✅")
except Exception as e:
    conn.rollback()  # Si hay error, cancelar transacción
    print("Error al crear el trigger tablas:", e)
finally:
    cur.close()
    conn.close()


Conexión creada con éxito
Tablas creadas correctamente ✅


In [4]:
conn = s_sql.crear_conexion()
cur = conn.cursor()

# Asegurar que no haya transacciones bloqueadas
conn.rollback()  

crear_vectorial = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE documentos (
    id SERIAL PRIMARY KEY,
    contenido TEXT,
    embedding vector(1536)  
);
"""

try:
    cur.execute(crear_vectorial)
    conn.commit()
    print("Tablas creadas correctamente ✅")
except Exception as e:
    conn.rollback()  # Si hay error, cancelar transacción
    print("Error al crear las tablas:", e)
finally:
    cur.close()
    conn.close()

Conexión creada con éxito
Tablas creadas correctamente ✅


Creamos el trigger de SQL para hacer los escrapeos automáticos

In [51]:
conn = s_sql.crear_conexion()
cur = conn.cursor()

# Asegurar que no haya transacciones bloqueadas
conn.rollback()  

crear_trigger = """
CREATE OR REPLACE FUNCTION notificar_modelo() RETURNS TRIGGER AS $$

BEGIN
  PERFORM pg_notify('scraping_channel', row_to_json(NEW)::text);
  RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER scraping_trigger
AFTER INSERT ON modelos

FOR EACH ROW EXECUTE FUNCTION notificar_modelo();

"""

try:
    cur.execute(crear_trigger)
    conn.commit()
    print("Trigger creado correctamente ✅")
except Exception as e:
    conn.rollback()  # Si hay error, cancelar transacción
    print("Error al crear el trigger:", e)
finally:
    cur.close()
    conn.close()


Trigger creado correctamente ✅


En primer lugar lo que haremos será buscar todas las URLs de los coches a escrapear. Contruimos un df con las mismas y lo subimos directamente a Dbeaver

In [ ]:
##Km77 comenzó sus reviews con coches a partir de 1999, por lo que limitamos la búsqueda al número máximo de páginas.
conn = s_sql.crear_conexion()

cur = conn.cursor()
url_busqueda = "https://www.km77.com/buscador/informaciones?grouped=0&order=date-desc&markets[]=current&markets[]=discontinued"
res_busqueda = requests.get(url_busqueda)
sopa_busqueda = BeautifulSoup(res_busqueda.text,"html.parser")
pags_totales = sopa_busqueda.find("div", class_="d-inline font-weight-normal font-size-2xl").text.strip()
total_resultados = int(re.findall(r"\d+",pags_totales)[0])

for numero_pagina in range(1,(total_resultados//20)+1):
        sleep(5)
        url_review = f"https://www.km77.com/buscador/informaciones?grouped=0&order=date-desc&markets[]=current&markets[]=discontinued&page={numero_pagina}"
        res_review = requests.get(url_review)
        print(res_review)

        sopa_review = BeautifulSoup(res_review.content, "html.parser")
        titulo = sopa_review.find("div", class_="d-inline font-weight-normal font-size-2xl")
        if titulo.text != '(0 resultados)':

                if res_review.status_code == 200:
                        try:
                                print(f"Página {numero_pagina} captada correctamente")
                                
                                links = [link["href"] for link in sopa_review.findAll("a", class_="font-size-sm", href=True)]
                                links = set(links)

                                for enlace in (links):
                                        dic_coche = {}
                                        if len(enlace.split("/")) == 7:
                                                        dic_coche = [{
                                                        "marca": enlace.split("/")[2],
                                                        "modelo": enlace.split("/")[3],
                                                        "anio": enlace.split("/")[4],
                                                        "tipo_carroceria": enlace.split("/")[5],
                                                        "link": enlace
                                                        }]
                                        elif len(enlace.split("/")) == 8:

                                                        dic_coche = [{
                                                        "marca": enlace.split("/")[2],
                                                        "modelo": (enlace.split("/")[3] + " " + enlace.split("/")[6]),
                                                        "anio": enlace.split("/")[4],
                                                        "tipo_carroceria": enlace.split("/")[5],
                                                        "link": enlace
                                                        }]
                                        
                                        df = pd.DataFrame(dic_coche)
                                        df[["marca","modelo","anio","tipo_carroceria"]] = df[["marca","modelo","anio","tipo_carroceria"]].applymap(lambda x: s_limpieza.limpiar_texto(x))
                                        display(df)

                                        for row in df.itertuples(index=False):
                                                cur.execute("""
                                                INSERT INTO modelos (marca, modelo, anio, tipo_carroceria, link)
                                                VALUES (%s, %s, %s, %s, %s)
                                                ON CONFLICT (marca, modelo, anio)
                                                DO UPDATE SET
                                                tipo_carroceria = EXCLUDED.tipo_carroceria,
                                                link = EXCLUDED.link;
                                        """, (row.marca, row.modelo, row.anio, row.tipo_carroceria, row.link))
                                                conn.commit()
                                                print(f"Fila {df[['modelo']]} insertada correctamente")
                                                sleep(1)
                        except KeyError as e:
                                pass
                                
                                                
                                
                else:
                        print(f"ERROR en la página {numero_pagina}")
                        pass

        else:             
                print("Sin resultados")   
                cur.close()
                conn.close()
                break
        


Una vez escrapeados los modelos vamos a ver las características principales de cada uno:

In [9]:
conn = s_sql.crear_conexion()
query = '''select marca,modelo,anio, link 
from public.modelos c ;'''

cur = conn.cursor()

cur.execute(query)
conn.commit()

model_ids = cur.fetchall()
conn.close()

Conexión creada con éxito


In [ ]:
conn = s_sql.crear_conexion()
conn.rollback()
for datos_coche in model_ids:

    url_coche = f"https://www.km77.com{datos_coche[-1]}"
    print(url_coche)
    
    res_coche = requests.get(url_coche)
    print(res_coche.status_code)
    if res_coche.status_code == 200:
        sopa_carro = BeautifulSoup(res_coche.content,"html.parser")
        tablas = sopa_carro.find_all("table", class_="table table-sm table-hover mb-1")

        if tablas == []:
            pass
        else:
                #Esto es para sacar los numeros de la tabla que el formato es un poco extraño
            lista_caracteristicas = [row.get_text() for row in tablas[0].find_all("tr")]

            if s_limpieza.buscar_palabra(lista_caracteristicas,"Longitud"):
                    result_long = lista_caracteristicas[s_limpieza.buscar_palabra(lista_caracteristicas,"Longitud")]
                    longitud = re.findall(r"\d+,\d+", result_long)
            else:
                    longitud = None

            if s_limpieza.buscar_palabra(lista_caracteristicas,"maletero"):
                    result_maletero = lista_caracteristicas[s_limpieza.buscar_palabra(lista_caracteristicas,"maletero")]
                    maletero = re.findall(r"\d+",result_maletero)
            else:
                        maletero = None


            if s_limpieza.buscar_palabra(lista_caracteristicas,"Potencia"):
                            potencia = lista_caracteristicas[s_limpieza.buscar_palabra(lista_caracteristicas,"Potencia")]
                            pot_min =re.findall(r"\d+",potencia)[0]
                            pot_max =re.findall(r"\d+",potencia)[-1]
            else:
                        pot_max = None
                        pot_min = None

            if s_limpieza.buscar_palabra(lista_caracteristicas,"Caja de cambios"):
                            cambios = lista_caracteristicas[s_limpieza.buscar_palabra(lista_caracteristicas,"Caja de cambios")]
                            caja_cambios = re.findall(r"Man|Aut", cambios)
            else:
                        caja_cambios = None

            if s_limpieza.buscar_palabra(lista_caracteristicas,"Tracción"):
                            tracc = lista_caracteristicas[s_limpieza.buscar_palabra(lista_caracteristicas,"Tracción")]
                            tracción = re.split(r"n", tracc)[1]
            else:
                        tracción = None
                        
            if s_limpieza.buscar_palabra(lista_caracteristicas,"Combustible"):
                            gasofa = lista_caracteristicas[s_limpieza.buscar_palabra(lista_caracteristicas,"Combustible")]
                            combustible = re.findall(r"[G|D]", gasofa)
            else:
                        combustible = None
            

            dic_info_carro = [{
                    
                    "Precio_base": [row.get_text() for row in tablas[0].find_all("tr")][0].split("\n")[1].split("€")[0],
                    "Precio_max": [row.get_text() for row in tablas[0].find_all("tr")][0].split("\n")[2].split("€")[0],
                    "Fecha_inicio": [row.get_text() for row in tablas[0].find_all("tr")][2].split("\n")[1].split("-")[0],
                    "Fecha_fin": [row.get_text() for row in tablas[0].find_all("tr")][2].split("\n")[1].split("-")[-1],
                    "Longitud": longitud,
                    "Volumen_maletero": maletero,
                    "Potencia_minima": pot_min,
                    "Potencia_maxima": pot_max,
                    "Caja_cambios": caja_cambios,
                    "Tipo_traccion": tracción,
                    "Combustible": [combustible if len(tablas[0].find_all("tr")) == 11 else "Hibrido/Eléctrico"]
                    }]
            
            df_info_carro = pd.DataFrame(dic_info_carro)

            df_info_carro = s_limpieza.normalizar_columnas_caracteristicas(df_info_carro)
            df_info_carro = s_limpieza.expandir_columnas_caracteristicas(df_info_carro)
            df_info_carro[["marca","modelo","anio"]] = (datos_coche[0],datos_coche[1],datos_coche[2])
            df_info_carro = s_limpieza.estandarizar_df_caracteristicas(df_info_carro)
        



            cur = conn.cursor()

            for row in df_info_carro.itertuples(index=False):
                    try:
                        cur.execute("""
                        INSERT INTO caracteristicas ("marca","modelo","anio","precio_base","precio_max","fecha_inicio","fecha_fin","potencia_minima","potencia_maxima","lon_0","lon_1","maletero_0","maletero_1","cambio_man","cambio_aut","tracc_del","tracc_total","tracc_tras","diesel","gasolina","electrificado")
                        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                    
                        ON CONFLICT (marca, modelo, anio)
                        DO UPDATE SET

                        precio_base = EXCLUDED.precio_base,
                        precio_max = EXCLUDED.precio_max,
                        fecha_inicio = EXCLUDED.fecha_inicio,
                        fecha_fin = EXCLUDED.fecha_fin,
                        potencia_minima = EXCLUDED.potencia_minima,
                        potencia_maxima = EXCLUDED.potencia_maxima,
                        lon_0 = EXCLUDED.lon_0,
                        lon_1 = EXCLUDED.lon_1,
                        maletero_0 = EXCLUDED.maletero_0,
                        maletero_1 = EXCLUDED.maletero_1,
                        cambio_man = EXCLUDED.cambio_man,
                        cambio_aut = EXCLUDED.cambio_aut,
                        tracc_del = EXCLUDED.tracc_del,
                        tracc_total = EXCLUDED.tracc_total,
                        tracc_tras = EXCLUDED.tracc_tras,
                        diesel = EXCLUDED.diesel,
                        gasolina = EXCLUDED.gasolina,
                        electrificado = EXCLUDED.electrificado;                                    
                        """, 
                        (row.marca,
                            row.modelo,
                            row.anio,
                            row.Precio_base if type(row.Precio_base) == int else 0,
                            row.Precio_max if type(row.Precio_max) == int else 0,
                            row.Fecha_inicio if pd.notna(row.Fecha_inicio) else 0,
                            row.Fecha_fin if pd.notna(row.Fecha_fin) else 0,
                            row.Potencia_minima if pd.notna(row.Potencia_minima) else 0,
                            row.Potencia_maxima if pd.notna(row.Potencia_maxima) else 0,
                            row.Lon_0 if pd.notna(row.Lon_0) else 0,
                            row.Lon_1 if pd.notna(row.Lon_1) else 0,
                            row.Maletero_0 if pd.notna(row.Maletero_0) else 0,
                            row.Maletero_1 if pd.notna(row.Maletero_1) else 0,
                            row.Cambio_man if pd.notna(row.Cambio_man) else 0,
                            row.Cambio_aut if pd.notna(row.Cambio_aut) else 0,
                            row.Tracc_del if pd.notna(row.Tracc_del) else 0,
                            row.Tracc_total if pd.notna(row.Tracc_total) else 0,
                            row.Tracc_tras if pd.notna(row.Tracc_tras) else 0,
                            row.Diesel if pd.notna(row.Diesel) else 0,
                            row.Gasolina if pd.notna(row.Gasolina) else 0,
                            row.Electrificado if pd.notna(row.Electrificado) else 0)),
                        
                        conn.commit()
                        print(f"Fila {row.modelo} insertada correctamente")
                        sleep(2)

                    except NameError:
                        pass
             
            

      
    else:
        print(f"Este no arranca ({model_ids[-1]})")
        conn.close()
        break

Conexión creada con éxito
https://www.km77.com/coches/jaecoo/7/2024/estandar/informacion
200
Fila 7 insertada correctamente
https://www.km77.com/coches/ford/kuga/2024/estandar/informacion
200
Fila Kuga insertada correctamente
https://www.km77.com/coches/lynk-co/08/2024/estandar/informacion
200
https://www.km77.com/coches/volvo/es90/2025/estandar/informacion
200
https://www.km77.com/coches/audi/q4/2021/estandar/informacion
200
Fila Q4 insertada correctamente
https://www.km77.com/coches/renault/megane/2022/berlina/informacion
200
Fila Megane insertada correctamente
https://www.km77.com/coches/swm/g03/2025/estandar/informacion
200
Fila G03 insertada correctamente
https://www.km77.com/coches/mitsubishi/grandis/2026/estandar/informacion
200
https://www.km77.com/coches/xpeng/g6/2025/estandar/informacion
200
Fila G6 insertada correctamente
https://www.km77.com/coches/alfa-romeo/junior/2025/estandar/informacion
200
Fila Junior insertada correctamente
https://www.km77.com/coches/bmw/serie-2/202

KeyboardInterrupt: 

In [ ]:
df_info_carro

,marca,modelo,anio,Precio_base,Precio_max,Fecha_inicio,Fecha_fin,Potencia_minima,Potencia_maxima,Lon_0,Lon_1,Maletero_0,Maletero_1,Cambio_man,Cambio_aut,Tracc_del,Tracc_total,Tracc_tras,Diesel,Gasolina,Electrificado
0,Leapmotor,C10,2024,32824,35074,2024-09-01,NaT,215,218,4.74,NaN,400,435,NaN,1,NaN,NaN,1,NaN,NaN,NaN


In [ ]:
model_ids

[('Jaecoo', '7', 2024, '/coches/jaecoo/7/2024/estandar/informacion'),
 ('Bmw', 'Serie_2', 2025, '/coches/bmw/serie-2/2025/gran-coupe/informacion'),
 ('Xpeng', 'G6', 2025, '/coches/xpeng/g6/2025/estandar/informacion'),
 ('Ford', 'Kuga', 2024, '/coches/ford/kuga/2024/estandar/informacion'),
 ('Mitsubishi',
  'Grandis',
  2026,
  '/coches/mitsubishi/grandis/2026/estandar/informacion'),
 ('Volvo', 'Xc60', 2025, '/coches/volvo/xc60/2025/estandar/informacion'),
 ('Audi', 'A6', 2025, '/coches/audi/a6/2025/sportback/informacion'),
 ('Volvo', 'Ex90', 2023, '/coches/volvo/ex90/2023/estandar/informacion'),
 ('Kia', 'Ev4', 2025, '/coches/kia/ev4/2025/estandar/informacion'),
 ('Toyota',
  'Yaris_cross',
  2021,
  '/coches/toyota/yaris-cross/2021/estandar/informacion'),
 ('Swm', 'G03', 2025, '/coches/swm/g03/2025/estandar/informacion'),
 ('Tesla', 'Model_y', 2025, '/coches/tesla/model-y/2025/estandar/informacion'),
 ('Cupra', 'Leon', 2024, '/coches/cupra/leon/2024/5-puertas/informacion'),
 ('Alfa_ro

Extraccion de las descripciones

In [13]:
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--headless")  # evita que se abrán las ventanas del driver
driver = webdriver.Chrome(options=chrome_options)
for datos_coche in model_ids:
    url = f"https://www.km77.com{datos_coche[-1]}"
    print(url)
    #entramos en la pagina
    driver.get(url)
    # Maximizamos a pantalla completa
    driver.maximize_window()

    # Encontrar y hacer clic en el botón de aceptar cookies
    try:
        driver.find_element("xpath", "/html/body/div[1]/div/div/div/div/div/div[3]/button[2]").click()
        print("Botón de aceptar cookies clicado con éxito.")
    except:
        print("No se pudo encontrar el botón de aceptar cookies o hacer clic en él.")
        pass

    #Aquí vamos a coger toda la información de la descripción
    # driver.execute_script("window. scrollTo(0, document. body. scrollHeight);")
    sleep(5)
    # #Cogemos todo el los párrafos que sean texto
    try:
        spans = driver.find_elements(By.CLASS_NAME, "texto")
        lista_textos = []
        for span in spans:
                lista_textos.append(span.text)
                print("textos extraídos con éxito")

    except:
        pass

    lista_textos = []

    rurl = requests.get(url)
    sopa_text = BeautifulSoup(rurl.content, 'html.parser')

    # textos_fortalezas = soup.find_all('strong')
    # lista_fortalezas.append([texto.get_text() for texto in textos_fortalezas])

    div_content = sopa_text.find_all('div', class_='mainbar col-12 col-lg min-width-0')
    lista_textos.append([texto.get_text() for texto in div_content])

    lista_textos = ",".join(lista_textos[0])

        

    ##Insertamos en SQL
    conn = s_sql.crear_conexion()
    cur = conn.cursor()
    list_textos = ",".join(lista_textos)
    dic_texto = {"marca": datos_coche[0],
                 "modelo": datos_coche[1],
                 "anio":datos_coche[2],
                 "reportaje":lista_textos}
    

    try:
            cur.execute("""
            INSERT INTO reviews(marca,modelo,anio,reportaje)
            VALUES(%s,%s,%s,%s)
            ON CONFLICT (marca,modelo,anio) DO UPDATE SET
            reportaje = EXCLUDED.reportaje""",(dic_texto.get("marca"),dic_texto.get("modelo"),dic_texto.get("anio"),dic_texto.get("reportaje")))

            conn.commit()
            print(f"Fila {'modelo'} insertada correctamente")

            sleep(10)
    except NameError as e:
            pass                
    
driver.quit()
conn.close()    


https://www.km77.com/coches/jaecoo/7/2024/estandar/informacion
No se pudo encontrar el botón de aceptar cookies o hacer clic en él.
Conexión creada con éxito
Fila modelo insertada correctamente
https://www.km77.com/coches/ford/kuga/2024/estandar/informacion
No se pudo encontrar el botón de aceptar cookies o hacer clic en él.
Conexión creada con éxito
Fila modelo insertada correctamente
https://www.km77.com/coches/lynk-co/08/2024/estandar/informacion
No se pudo encontrar el botón de aceptar cookies o hacer clic en él.
Conexión creada con éxito
Fila modelo insertada correctamente
https://www.km77.com/coches/volvo/es90/2025/estandar/informacion
No se pudo encontrar el botón de aceptar cookies o hacer clic en él.
Conexión creada con éxito
Fila modelo insertada correctamente
https://www.km77.com/coches/audi/q4/2021/estandar/informacion
No se pudo encontrar el botón de aceptar cookies o hacer clic en él.
Conexión creada con éxito
Fila modelo insertada correctamente
https://www.km77.com/coche

KeyboardInterrupt: 